In [1]:
# IMPORTS

import json
import math
import random
import copy
import time
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, cohen_kappa_score, roc_auc_score,
                             recall_score, confusion_matrix)
from tqdm.auto import tqdm
from scipy.stats import mannwhitneyu

print(torch.__version__, torch.cuda.is_available())

In [2]:
# MODEL AND DATA DEFINITIONS
#   PatchEmbedding, SingleScaleBoundaryEstimator,
#   MultiScaleChangepointModule, ContrastiveBoundaryModule,
#   RegimeStructuredAttention, MultiResolutionEncoder,
#   MultiResContrastiveNeuroState, MultiEpochDataset,
#   PseudoBoundaryLoss, AttentionPriorLoss, ACBLLoss,
#   forward_with_intermediates, create_acbl_dataloaders

# Unchanged from ACBL_and_Gradient_Isolation so the multi seed
# results correspond to the same architecture.

class PatchEmbedding(nn.Module):
    def __init__(self, n_channels=22, n_samples=3000, embed_dim=128,
                 temporal_kernel=25, pool_kernel=75, pool_stride=15,
                 dropout=0.1):
        super().__init__()
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, 40, (1, temporal_kernel),
                      padding=(0, temporal_kernel // 2)),
            nn.BatchNorm2d(40), nn.GELU())
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(40, 40, (n_channels, 1)),
            nn.BatchNorm2d(40), nn.GELU())
        self.pool = nn.AvgPool2d((1, pool_kernel), stride=(1, pool_stride))
        self.projection = nn.Sequential(
            nn.Conv2d(40, embed_dim, (1, 1)), nn.Dropout(dropout))
        self.seq_len = (n_samples - pool_kernel) // pool_stride + 1

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.pool(x)
        x = self.projection(x)
        return x.squeeze(2).permute(0, 2, 1)

class SingleScaleBoundaryEstimator(nn.Module):
    def __init__(self, embed_dim, hidden_dim, kernel_size, dropout=0.1):
        super().__init__()
        ks = kernel_size if kernel_size % 2 == 1 else kernel_size + 1
        pad = ks // 2
        self.net = nn.Sequential(
            nn.Conv1d(embed_dim, hidden_dim, ks, padding=pad),
            nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, ks, padding=pad),
            nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, 1, 1))

    def forward(self, x):
        h = self.net(x.permute(0, 2, 1))
        return torch.sigmoid(h.squeeze(1))

class MultiScaleChangepointModule(nn.Module):
    def __init__(self, embed_dim=128, hidden_dim=64,
                 scales=(3, 13, 65), dropout=0.1,
                 lambda_sparse=0.02, lambda_sharp=0.02):
        super().__init__()
        self.scales = scales
        self.n_scales = len(scales)
        self.lambda_sparse = lambda_sparse
        self.lambda_sharp = lambda_sharp
        self.estimators = nn.ModuleList([
            SingleScaleBoundaryEstimator(embed_dim, hidden_dim, k, dropout)
            for k in scales])
        self.fusion = nn.Sequential(
            nn.Linear(self.n_scales, self.n_scales * 2), nn.GELU(),
            nn.Linear(self.n_scales * 2, 1))

    def forward(self, x):
        per_scale = [est(x) for est in self.estimators]
        stacked = torch.stack(per_scale, dim=-1)
        fused = torch.sigmoid(self.fusion(stacked).squeeze(-1))
        boundary_loss = self._boundary_loss(fused, per_scale)
        return {'boundaries': fused, 'per_scale': per_scale,
                'boundary_loss': boundary_loss}

    def _boundary_loss(self, fused, per_scale):
        sparsity = fused.mean()
        eps = 1e-7
        entropy = -(fused * torch.log(fused + eps) +
                    (1 - fused) * torch.log(1 - fused + eps))
        sharpness = entropy.mean()
        consistency = sum(F.mse_loss(ps, fused.detach())
                         for ps in per_scale) / len(per_scale)
        return (self.lambda_sparse * sparsity +
                self.lambda_sharp * sharpness + 0.01 * consistency)

class ContrastiveBoundaryModule(nn.Module):
    """Boundaries from representation contrast, not absolute MLP."""

    def __init__(self, embed_dim=128, hidden_dim=64,
                 scales=(1, 4, 16), dropout=0.1):
        super().__init__()
        self.scales = scales
        self.n_scales = len(scales)
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, hidden_dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
            for _ in scales])
        self.fusion = nn.Sequential(
            nn.Linear(self.n_scales, self.n_scales * 2), nn.GELU(),
            nn.Linear(self.n_scales * 2, 1))
        self.temperature = nn.Parameter(torch.tensor(1.0))

    def _compute_contrast(self, x, proj, offset):
        B, T, D = x.shape
        h = proj(x)
        h_norm = F.normalize(h, dim=-1)
        if offset < T:
            h_shifted = torch.roll(h_norm, -offset, dims=1)
            h_shifted[:, -offset:, :] = h_norm[:, -offset:, :]
            similarity = (h_norm * h_shifted).sum(dim=-1)
            contrast = 1.0 - (similarity + 1.0) / 2.0
            contrast = torch.sigmoid(
                (contrast - 0.5) * self.temperature.abs().clamp(min=0.1))
        else:
            contrast = torch.zeros(B, T, device=x.device)
        return contrast

    def forward(self, x):
        per_scale = [self._compute_contrast(x, proj, off)
                     for proj, off in zip(self.projections, self.scales)]
        stacked = torch.stack(per_scale, dim=-1)
        fused = torch.sigmoid(self.fusion(stacked).squeeze(-1))
        consistency = sum(F.mse_loss(ps, fused.detach())
                         for ps in per_scale) / len(per_scale)
        return {'boundaries': fused, 'per_scale': per_scale,
                'boundary_loss': 0.01 * consistency}

class RegimeStructuredAttention(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_intra = n_intra
        self.n_inter = n_inter
        self.n_cross = n_cross
        self.n_heads = n_intra + n_inter + n_cross
        self.head_dim = embed_dim // self.n_heads
        assert embed_dim % self.n_heads == 0
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5

    def _build_regime_mask(self, boundaries):
        cum = torch.cumsum(boundaries, dim=1)
        return torch.exp(-torch.abs(cum.unsqueeze(2) - cum.unsqueeze(1)))

    def forward(self, x, boundaries, return_attention=False):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        same = self._build_regime_mask(boundaries).unsqueeze(1)
        cross = 1.0 - same
        h_ie = self.n_intra
        h_ce = h_ie + self.n_inter
        mask = torch.ones_like(attn)
        mask[:, :h_ie] = same.expand(B, self.n_intra, T, T)
        mask[:, h_ie:h_ce] = cross.expand(B, self.n_inter, T, T)
        attn = attn + torch.log(mask + 1e-6)
        attn_w = F.softmax(attn, dim=-1)
        attn_w = self.attn_drop(attn_w)
        out = (attn_w @ v).transpose(1, 2).reshape(B, T, D)
        out = self.proj_drop(self.out_proj(out))
        return (out, attn_w) if return_attention else out

class NeuroStateBlock(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = RegimeStructuredAttention(
            embed_dim, n_intra, n_inter, n_cross, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout))

    def forward(self, x, boundaries, return_attention=False):
        normed = self.norm1(x)
        if return_attention:
            a, w = self.attn(normed, boundaries, True)
            x = x + a
            x = x + self.mlp(self.norm2(x))
            return x, w
        x = x + self.attn(normed, boundaries)
        x = x + self.mlp(self.norm2(x))
        return x

class MultiResolutionEncoder(nn.Module):
    """Encodes a single epoch at 100/50/25 Hz, merges via interpolation."""

    def __init__(self, n_channels=3, n_samples=3000,
                 embed_dim=128, dropout=0.1):
        super().__init__()
        self.enc_100hz = PatchEmbedding(
            n_channels, n_samples, embed_dim, dropout=dropout)
        self.enc_50hz = PatchEmbedding(
            n_channels, n_samples // 2, embed_dim, dropout=dropout)
        self.enc_25hz = PatchEmbedding(
            n_channels, n_samples // 4, embed_dim, dropout=dropout)
        self.merge = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim), nn.GELU(),
            nn.Dropout(dropout))
        self.seq_len_100 = self.enc_100hz.seq_len
        self.seq_len_50 = self.enc_50hz.seq_len
        self.seq_len_25 = self.enc_25hz.seq_len

    def forward(self, x):
        x_50 = x[:, :, ::2]
        x_25 = x[:, :, ::4]
        emb_100 = self.enc_100hz(x)
        emb_50 = self.enc_50hz(x_50)
        emb_25 = self.enc_25hz(x_25)
        T1 = emb_100.shape[1]
        emb_50_up = F.interpolate(
            emb_50.permute(0, 2, 1), size=T1,
            mode='linear', align_corners=False).permute(0, 2, 1)
        emb_25_up = F.interpolate(
            emb_25.permute(0, 2, 1), size=T1,
            mode='linear', align_corners=False).permute(0, 2, 1)
        merged = torch.cat([emb_100, emb_50_up, emb_25_up], dim=-1)
        return self.merge(merged)

class MultiResContrastiveNeuroState(nn.Module):
    """MultiRes MultiEpoch NeuroState with contrastive boundaries."""

    def __init__(self, n_channels=3, n_samples=3000, n_classes=5,
                 embed_dim=128, n_layers=4, dropout=0.1,
                 n_intra=4, n_inter=2, n_cross=2,
                 contrast_scales=(1, 4, 16), cp_hidden=64,
                 n_context_epochs=3):
        super().__init__()
        self.n_classes = n_classes
        self.n_context = n_context_epochs
        self.mr_encoder = MultiResolutionEncoder(
            n_channels, n_samples, embed_dim, dropout)
        tokens_per_epoch = self.mr_encoder.seq_len_100
        total_tokens = tokens_per_epoch * n_context_epochs
        self.pos_embed = nn.Parameter(
            torch.randn(1, total_tokens, embed_dim) * 0.02)
        self.pos_drop = nn.Dropout(dropout)
        self.epoch_embed = nn.Parameter(
            torch.randn(1, n_context_epochs, 1, embed_dim) * 0.02)
        self.changepoint_module = ContrastiveBoundaryModule(
            embed_dim=embed_dim, hidden_dim=cp_hidden,
            scales=contrast_scales, dropout=dropout)
        self.blocks = nn.ModuleList([
            NeuroStateBlock(embed_dim, n_intra, n_inter, n_cross,
                            dropout=dropout)
            for _ in range(n_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, n_classes))
        self.tokens_per_epoch = tokens_per_epoch
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"ContrastiveMultiRes: {n_params/1e6:.2f}M params, "
              f"{total_tokens} tokens")

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x, return_boundaries=False):
        B, N, C, T = x.shape
        epoch_embs = []
        for i in range(N):
            emb = self.mr_encoder(x[:, i])
            emb = emb + self.epoch_embed[:, i]
            epoch_embs.append(emb)
        full_seq = torch.cat(epoch_embs, dim=1)
        full_seq = self.pos_drop(full_seq + self.pos_embed)
        cp_out = self.changepoint_module(full_seq)
        boundaries = cp_out['boundaries']
        boundary_loss = cp_out['boundary_loss']
        for block in self.blocks:
            full_seq = block(full_seq, boundaries)
        full_seq = self.norm(full_seq)
        start = self.tokens_per_epoch * (N // 2)
        end = start + self.tokens_per_epoch
        center_tokens = full_seq[:, start:end, :]
        pooled = center_tokens.mean(dim=1)
        logits = self.head(pooled)
        out = {'logits': logits, 'boundary_loss': boundary_loss}
        if return_boundaries:
            out['boundaries'] = boundaries
            out['per_scale'] = cp_out['per_scale']
        return out

class MultiEpochDataset(Dataset):
    """Returns 3 consecutive epochs. Label = center epoch."""

    def __init__(self, h5_path, task='sleep_staging',
                 target_channels=3, target_samples=3000,
                 indices=None, context_size=1):
        self.h5_path = Path(h5_path)
        self.target_channels = target_channels
        self.target_samples = target_samples
        self.context_size = context_size
        self._h5_file = None

        with h5py.File(self.h5_path, 'r') as f:
            self.total_samples = f['epochs'].shape[0]
            self.labels = f['labels'][:]
            self.subject_ids = f['subject_ids'][:]

        if isinstance(self.subject_ids[0], (bytes, np.bytes_)):
            self.subject_ids = np.array([
                s.decode() if isinstance(s, bytes) else s
                for s in self.subject_ids])

        base_indices = indices if indices is not None else np.arange(
            self.total_samples)
        base_set = set(base_indices)
        self.valid_indices = []

        for idx in base_indices:
            valid = True
            for offset in range(-context_size, context_size + 1):
                neighbor = idx + offset
                if neighbor < 0 or neighbor >= self.total_samples:
                    valid = False; break
                if neighbor not in base_set:
                    valid = False; break
                if self.subject_ids[neighbor] != self.subject_ids[idx]:
                    valid = False; break
            if valid:
                self.valid_indices.append(idx)

        self.valid_indices = np.array(self.valid_indices)
        self._compute_class_weights()
        dist = dict(zip(*np.unique(
            self.labels[self.valid_indices], return_counts=True)))
        print(f"  MultiEpoch: {len(self.valid_indices)} seqs, "
              f"classes={dist}")

    def _get_h5(self):
        if self._h5_file is None:
            self._h5_file = h5py.File(self.h5_path, 'r')
        return self._h5_file

    def _compute_class_weights(self):
        sub = self.labels[self.valid_indices]
        cls, cnt = np.unique(sub, return_counts=True)
        w = 1.0 / cnt; w /= w.sum()
        self.class_weights = dict(zip(cls, w))
        self.sample_weights = np.array(
            [self.class_weights[l] for l in sub])

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        center_idx = self.valid_indices[idx]
        f = self._get_h5()
        epochs = []
        epoch_labels = []
        for offset in range(-self.context_size,
                            self.context_size + 1):
            ep = f['epochs'][center_idx + offset]
            ep = self._standardize_shape(ep)
            epochs.append(ep)
            epoch_labels.append(self.labels[center_idx + offset])

        return {
            'epoch': torch.tensor(np.stack(epochs, axis=0),
                                  dtype=torch.float32),
            'label': torch.tensor(self.labels[center_idx],
                                  dtype=torch.long),
            'epoch_labels': torch.tensor(epoch_labels,
                                         dtype=torch.long),
        }

    def _standardize_shape(self, epoch):
        nc, nt = epoch.shape
        if nc < self.target_channels:
            epoch = np.vstack(
                [epoch, np.zeros((self.target_channels - nc, nt))])
        elif nc > self.target_channels:
            epoch = epoch[:self.target_channels]
        nc = self.target_channels
        if nt < self.target_samples:
            epoch = np.hstack(
                [epoch, np.zeros((nc, self.target_samples - nt))])
        elif nt > self.target_samples:
            s = (nt - self.target_samples) // 2
            epoch = epoch[:, s:s + self.target_samples]
        return epoch

    def get_sampler(self):
        return WeightedRandomSampler(
            self.sample_weights, len(self), True)

class PseudoBoundaryLoss(nn.Module):
    """
    Prong 1: Self supervised boundary signal from encoder representations.

    Computes cosine distance between adjacent tokens. High distance
    means representational shift, used as soft target for boundary head.
    Targets are detached to prevent encoder from trivially maximizing
    adjacent dissimilarity.
    """

    def __init__(self, temperature=2.0, tokens_per_epoch=196, n_epochs=3):
        super().__init__()
        self.temperature = temperature
        self.tokens_per_epoch = tokens_per_epoch
        self.n_epochs = n_epochs

        T = tokens_per_epoch * n_epochs
        mask = torch.ones(T - 1)
        for e in range(1, n_epochs):
            mask[e * tokens_per_epoch - 1] = 0.0
        self.register_buffer('epoch_boundary_mask', mask)

    def compute_pseudo_targets(self, h):
        """h: [B, 588, 128] encoder output. Returns [B, 588] soft targets."""
        B, T, D = h.shape
        h_det = h.detach()

        h_norm = F.normalize(h_det, dim=-1)
        cos_sim = (h_norm[:, :-1] * h_norm[:, 1:]).sum(dim=-1)
        cos_dist = (1.0 - cos_sim) * self.epoch_boundary_mask.to(h.device)

        d_min = cos_dist.min(dim=-1, keepdim=True).values
        d_max = cos_dist.max(dim=-1, keepdim=True).values
        d_range = (d_max - d_min).clamp(min=1e-6)
        cos_dist_norm = (cos_dist - d_min) / d_range

        pseudo = torch.sigmoid((cos_dist_norm - 0.5) * self.temperature)
        return F.pad(pseudo, (0, 1), mode='replicate')

    def forward(self, boundary_probs, h):
        pseudo_targets = self.compute_pseudo_targets(h)
        return F.binary_cross_entropy(
            boundary_probs.clamp(1e-6, 1 - 1e-6),
            pseudo_targets, reduction='mean')

class AttentionPriorLoss(nn.Module):
    """
    Prong 2: Label derived boundary and attention supervision.

    2A: Gaussian peaks at epoch junctions where labels change.
        Direct BCE on boundary head output.
    2B: KL divergence between regime attention and a block diagonal
        prior derived from epoch labels. Gives gradient through
        the attention mechanism back to the boundary head.
    """

    def __init__(self, tokens_per_epoch=196, n_epochs=3,
                 sigma=5.0, attn_prior_weight=0.5):
        super().__init__()
        self.tokens_per_epoch = tokens_per_epoch
        self.n_epochs = n_epochs
        self.sigma = sigma
        self.attn_prior_weight = attn_prior_weight
        self.T = tokens_per_epoch * n_epochs

    def _build_boundary_targets(self, labels, device):
        B = labels.shape[0]
        T = self.T
        tpe = self.tokens_per_epoch

        trans_01 = (labels[:, 0] != labels[:, 1]).float()
        trans_12 = (labels[:, 1] != labels[:, 2]).float()
        has_transition = (trans_01 + trans_12) > 0

        positions = torch.arange(T, device=device, dtype=torch.float32)
        targets = torch.zeros(B, T, device=device)

        junction_01 = float(tpe) - 0.5
        junction_12 = float(2 * tpe) - 0.5

        gauss_01 = torch.exp(
            -0.5 * ((positions - junction_01) / self.sigma) ** 2)
        gauss_12 = torch.exp(
            -0.5 * ((positions - junction_12) / self.sigma) ** 2)

        targets += trans_01.unsqueeze(1) * gauss_01.unsqueeze(0)
        targets += trans_12.unsqueeze(1) * gauss_12.unsqueeze(0)

        t_max = targets.max(dim=-1, keepdim=True).values.clamp(min=1e-6)
        targets = targets / t_max
        targets = targets * has_transition.float().unsqueeze(1)

        return targets, has_transition

    def _build_attention_prior(self, labels, device):
        B = labels.shape[0]
        T = self.T
        tpe = self.tokens_per_epoch

        token_labels = torch.zeros(B, T, dtype=labels.dtype, device=device)
        for e in range(self.n_epochs):
            start = e * tpe
            end = (e + 1) * tpe
            token_labels[:, start:end] = labels[:, e].unsqueeze(1)

        same_stage = (token_labels.unsqueeze(2) ==
                      token_labels.unsqueeze(1)).float()
        prior = same_stage * 0.8 + (1 - same_stage) * 0.2
        prior = prior / prior.sum(dim=-1, keepdim=True)
        return prior

    def forward(self, boundary_probs, regime_attention, labels):
        device = boundary_probs.device
        boundary_targets, has_transition = self._build_boundary_targets(
            labels, device)
        n_transitions = has_transition.sum().item()

        results = {
            'n_transitions': n_transitions,
            'boundary_target_loss': torch.tensor(0.0, device=device),
            'attention_prior_loss': torch.tensor(0.0, device=device),
        }

        if n_transitions == 0:
            results['total'] = torch.tensor(0.0, device=device)
            return results

        mask = has_transition.float()
        bce = F.binary_cross_entropy(
            boundary_probs.clamp(1e-6, 1 - 1e-6),
            boundary_targets, reduction='none')
        bce_per_sample = bce.mean(dim=-1)
        results['boundary_target_loss'] = (
            (bce_per_sample * mask).sum() / mask.sum())

        if regime_attention is not None:
            if regime_attention.dim() == 4:
                attn = regime_attention.mean(dim=1)
            else:
                attn = regime_attention

            prior = self._build_attention_prior(labels, device)
            log_attn = torch.log(attn.clamp(min=1e-8))
            log_prior = torch.log(prior.clamp(min=1e-8))
            kl = prior * (log_prior - log_attn)
            kl_per_sample = kl.sum(dim=-1).mean(dim=-1)
            results['attention_prior_loss'] = (
                (kl_per_sample * mask).sum() / mask.sum())

        results['total'] = (
            results['boundary_target_loss'] +
            self.attn_prior_weight * results['attention_prior_loss'])
        return results

class ACBLLoss(nn.Module):
    """Combined ACBL loss with temperature annealing and variance penalty."""

    def __init__(self, tokens_per_epoch=196, n_epochs=3,
                 pseudo_weight=0.3, pseudo_temperature=2.0,
                 pseudo_temp_min=0.5, pseudo_temp_anneal_epochs=15,
                 prior_weight=0.5, sigma=5.0,
                 attn_prior_weight=0.5,
                 variance_weight=1.0):
        super().__init__()
        self.pseudo_weight = pseudo_weight
        self.prior_weight = prior_weight
        self.variance_weight = variance_weight
        self.pseudo_temp_min = pseudo_temp_min
        self.pseudo_temp_anneal_epochs = pseudo_temp_anneal_epochs
        self.pseudo_temperature_init = pseudo_temperature

        self.prong1 = PseudoBoundaryLoss(
            temperature=pseudo_temperature,
            tokens_per_epoch=tokens_per_epoch,
            n_epochs=n_epochs)

        self.prong2 = AttentionPriorLoss(
            tokens_per_epoch=tokens_per_epoch,
            n_epochs=n_epochs,
            sigma=sigma,
            attn_prior_weight=attn_prior_weight)

    def anneal_temperature(self, epoch):
        if self.pseudo_temp_anneal_epochs <= 0:
            return
        progress = min(epoch / self.pseudo_temp_anneal_epochs, 1.0)
        self.prong1.temperature = (
            self.pseudo_temperature_init -
            progress * (self.pseudo_temperature_init - self.pseudo_temp_min))

    def forward(self, boundary_probs, encoder_h, regime_attention,
                epoch_labels, epoch=0):
        self.anneal_temperature(epoch)
        pseudo_loss = self.prong1(boundary_probs, encoder_h)
        prior_results = self.prong2(
            boundary_probs, regime_attention, epoch_labels)

        # Prong 3: Variance penalty (anti collapse)
        # Negative variance = penalize constant outputs
        # Computed per sample then averaged
        var_per_sample = boundary_probs.var(dim=-1)  # [B]
        variance_loss = -var_per_sample.mean()

        total = (self.pseudo_weight * pseudo_loss +
                 self.prior_weight * prior_results['total'] +
                 self.variance_weight * variance_loss)
        return {
            'acbl_total': total,
            'pseudo_boundary_loss': pseudo_loss,
            'boundary_target_loss': prior_results['boundary_target_loss'],
            'attention_prior_loss': prior_results['attention_prior_loss'],
            'variance_loss': variance_loss,
            'boundary_variance': var_per_sample.mean(),
            'n_transitions': prior_results['n_transitions'],
            'pseudo_temperature': self.prong1.temperature,
        }

def forward_with_intermediates(model, x, detach_boundaries=False):
    """
    Drop in replacement for model.forward() that exposes:
      encoder_h:        embeddings before boundary detection
      boundary_probs:   boundary head output
      regime_attention:  attention weights from last transformer block

    When detach_boundaries=True, the boundaries are detached before
    entering regime attention. This means classification loss CANNOT
    send gradients back to the boundary head. The boundary head only
    receives gradients from ACBL losses. This prevents classification
    from crushing boundary diversity during the formation phase.

    Follows the exact flow from tier 2 MultiResContrastiveNeuroState.
    """
    B, N, C, T = x.shape

    epoch_embs = []
    for i in range(N):
        emb = model.mr_encoder(x[:, i])
        emb = emb + model.epoch_embed[:, i]
        epoch_embs.append(emb)

    full_seq = torch.cat(epoch_embs, dim=1)
    full_seq = model.pos_drop(full_seq + model.pos_embed)

    # This is encoder_h: before boundary detection
    encoder_h = full_seq

    cp_out = model.changepoint_module(full_seq)
    boundaries = cp_out['boundaries']
    boundary_loss = cp_out['boundary_loss']

    # Gradient isolation: detach boundaries from classification path
    # so only ACBL losses can update the boundary head
    boundaries_for_attn = boundaries.detach() if detach_boundaries else boundaries

    # Run transformer blocks, capture attention from last block
    regime_attention = None
    for i, block in enumerate(model.blocks):
        if i == len(model.blocks) - 1:
            full_seq, attn_w = block(
                full_seq, boundaries_for_attn, return_attention=True)
            regime_attention = attn_w
        else:
            full_seq = block(full_seq, boundaries_for_attn)

    full_seq = model.norm(full_seq)
    tpe = model.tokens_per_epoch
    start = tpe * (N // 2)
    end = start + tpe
    pooled = full_seq[:, start:end, :].mean(dim=1)
    logits = model.head(pooled)

    return {
        'logits': logits,
        'boundary_loss': boundary_loss,
        'boundaries': boundaries,
        'encoder_h': encoder_h,
        'boundary_probs': boundaries,
        'regime_attention': regime_attention,
    }

def create_acbl_dataloaders(h5_path, batch_size=16, seed=42):
    """Subject level split with 3 epoch label sequences."""
    h5_path = Path(h5_path)
    rng = np.random.RandomState(seed)

    with h5py.File(h5_path, 'r') as f:
        labels = f['labels'][:]
        subject_ids = f['subject_ids'][:]

    if isinstance(subject_ids[0], (bytes, np.bytes_)):
        subject_ids = np.array([
            s.decode() if isinstance(s, bytes) else s
            for s in subject_ids])

    unique_subjects = np.unique(subject_ids)
    n_subjects = len(unique_subjects)
    rng.shuffle(unique_subjects)

    n_train = int(0.7 * n_subjects)
    n_val = int(0.15 * n_subjects)

    train_subj = set(unique_subjects[:n_train])
    val_subj = set(unique_subjects[n_train:n_train + n_val])
    test_subj = set(unique_subjects[n_train + n_val:])

    train_idx = np.where(np.isin(subject_ids, list(train_subj)))[0]
    val_idx = np.where(np.isin(subject_ids, list(val_subj)))[0]
    test_idx = np.where(np.isin(subject_ids, list(test_subj)))[0]

    print(f"Split: {len(train_subj)} train, {len(val_subj)} val, "
          f"{len(test_subj)} test subjects")
    for name, idx in [('Train', train_idx), ('Val', val_idx),
                      ('Test', test_idx)]:
        dist = dict(zip(*np.unique(labels[idx], return_counts=True)))
        print(f"  {name}: {len(idx)} epochs, {dist}")

    train_ds = MultiEpochDataset(
        h5_path, indices=train_idx, context_size=1)
    val_ds = MultiEpochDataset(
        h5_path, indices=val_idx, context_size=1)
    test_ds = MultiEpochDataset(
        h5_path, indices=test_idx, context_size=1)

    train_ld = DataLoader(
        train_ds, batch_size, sampler=train_ds.get_sampler(),
        num_workers=0, drop_last=True)
    val_ld = DataLoader(
        val_ds, batch_size, shuffle=False, num_workers=0)
    test_ld = DataLoader(
        test_ds, batch_size, shuffle=False, num_workers=0)

    return train_ld, val_ld, test_ld


In [3]:
# SEEDING

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [4]:
# CONFIG

SLEEP_EDF_CONFIG = {
    'name': 'sleep_edf',
    'h5_path': 'data/processed/sleep_edf_processed.h5',
    'n_channels': 3,
    'n_samples': 3000,
    'n_classes': 5,
    'batch_size': 16,
    'split_seed': 42,
    'embed_dim': 128,
    'n_layers': 4,
    'dropout': 0.1,
    'n_intra': 4,
    'n_inter': 2,
    'n_cross': 2,
    'contrast_scales': (1, 4, 16),
    'cp_hidden': 64,
    'n_context_epochs': 3,
    'n_epochs': 50,
    'warmup_epochs': 3,
    'formation_epochs': 12,
    'patience': 12,
    'acbl_weight': 0.3,
    'cls_weight': 1.0,
    'pseudo_weight': 0.3,
    'prior_weight': 0.5,
    'sigma': 5.0,
    'pseudo_temperature': 2.0,
    'pseudo_temp_min': 0.5,
    'pseudo_temp_anneal_epochs': 15,
    'attn_prior_weight': 0.5,
    'variance_weight': 1.0,
    'lr_other': 1e-4,
    'lr_boundary': 3e-4,
    'wd_other': 1e-4,
    'wd_boundary': 1e-5,
    'binary': False,
    'loader': 'sleep_edf',
}

# CHB-MIT uses the combined per subject file, a 17 channel montage and
# batch size 32, and is loaded through the round robin split rather than
# the random subject shuffle used for Sleep-EDF.
CHB_MIT_CONFIG = dict(SLEEP_EDF_CONFIG)
CHB_MIT_CONFIG.update({
    'name': 'chb_mit',
    'h5_path': 'data/processed/chbmit_combined_seizure_detection.h5',
    'n_channels': 17,
    'n_classes': 2,
    'batch_size': 32,
    'binary': True,
    'loader': 'chbmit',
    'single_param_group': True,
    'class_weighted_loss': True,
})


In [7]:
import h5py
import numpy as np

with h5py.File('data/processed/sleep_edf_processed.h5', 'r') as f:
    print(list(f.keys()))
    labels = f['labels'][:]
    subj = f['subject_ids'][:]
    times = f['times'][:] if 'times' in f else None

if times is not None:
    s = np.array([x.decode() if isinstance(x, bytes) else x for x in subj])
    same = s[1:] == s[:-1]
    gaps = np.diff(times)[same]
    print('gap==30s:', float((np.abs(gaps - 30) < 1).mean()))
    print('median gap:', float(np.median(gaps)))

['bad_epochs_mask', 'epochs', 'labels', 'subject_ids', 'times']
gap==30s: 0.3352280604464162
median gap: 180.0


In [5]:
# TRANSITION DENSITY BY DATASET

def transition_density(loader):
    n_trans, n_total = 0, 0
    for batch in loader:
        el = batch['epoch_labels'].numpy()
        n_trans += int((el.max(axis=1) != el.min(axis=1)).sum())
        n_total += el.shape[0]
    return n_trans, n_total, n_trans / n_total

_, _, sleep_test = create_acbl_dataloaders(
    Path(SLEEP_EDF_CONFIG['h5_path']),
    batch_size=SLEEP_EDF_CONFIG['batch_size'],
    seed=SLEEP_EDF_CONFIG['split_seed'])
print('sleep_edf', transition_density(sleep_test))

Split: 102 train, 22 val, 23 test subjects
  Train: 5152 epochs, {0: 913, 1: 1397, 2: 1045, 3: 1701, 4: 96}
  Val: 1137 epochs, {0: 202, 1: 429, 2: 212, 3: 265, 4: 29}
  Test: 1071 epochs, {0: 205, 1: 333, 2: 191, 3: 325, 4: 17}
  MultiEpoch: 4948 seqs, classes={0: 865, 1: 1290, 2: 1029, 3: 1669, 4: 95}
  MultiEpoch: 1093 seqs, classes={0: 191, 1: 406, 2: 205, 3: 262, 4: 29}
  MultiEpoch: 1025 seqs, classes={0: 188, 1: 311, 2: 188, 3: 321, 4: 17}
sleep_edf (829, 1025, 0.8087804878048781)


In [9]:
# CONTIGUITY DIAGNOSTICS

def decode_ids(ids):
    return np.array([x.decode() if isinstance(x, (bytes, np.bytes_)) else x
                     for x in ids])


def inspect(h5_path):
    with h5py.File(h5_path, 'r') as f:
        print(h5_path)
        print('  keys:', list(f.keys()))
        print('  attrs:', dict(f.attrs))
        for k in f.keys():
            try:
                print(f'    {k}: {f[k].shape} {f[k].dtype}')
            except Exception:
                pass


def gap_test(h5_path, epoch_seconds=30.0):
    """Definitive contiguity test. Requires a stored onset time array."""
    with h5py.File(h5_path, 'r') as f:
        if 'times' not in f:
            print('  no times array, gap test not possible')
            return None
        times = f['times'][:]
        subj = decode_ids(f['subject_ids'][:])
    same = subj[1:] == subj[:-1]
    gaps = np.diff(times)[same]
    contiguous = np.abs(gaps - epoch_seconds) < 1.0
    print(f'  within-subject adjacent pairs: {int(same.sum())}')
    print(f'  fraction exactly one epoch apart: {contiguous.mean():.4f}')
    print(f'  median gap: {np.median(gaps):.1f}s')
    return same, contiguous


def seam_stats(h5_path, pair_idx):
    """Mean absolute step across each epoch join, and within each epoch."""
    seam, within = [], []
    with h5py.File(h5_path, 'r') as f:
        ep = f['epochs']
        for i in pair_idx:
            a = ep[i]
            b = ep[i + 1]
            seam.append(float(np.abs(b[:, 0] - a[:, -1]).mean()))
            within.append(float(np.abs(np.diff(a, axis=1)).mean()))
    return np.array(seam), np.array(within)


def seam_test(h5_path, max_pairs=2000, seed=42, known=None):
    """Signal based contiguity test for files with no stored times.

    A ratio near 1 means the join looks like an ordinary sample step,
    so the epochs are contiguous. A large ratio means the two epochs
    are unrelated in time.
    """
    rng = np.random.RandomState(seed)
    with h5py.File(h5_path, 'r') as f:
        n = f['epochs'].shape[0]
        subj = decode_ids(f['subject_ids'][:])
    same = subj[1:] == subj[:-1]
    idx = np.arange(n - 1)[same]
    if len(idx) > max_pairs:
        idx = np.sort(rng.choice(idx, max_pairs, replace=False))
    seam, within = seam_stats(h5_path, idx)
    ratio = seam / np.maximum(within, 1e-8)
    print(f'  pairs tested: {len(idx)}')
    print(f'  mean seam step: {seam.mean():.4f}')
    print(f'  mean within-epoch step: {within.mean():.4f}')
    print(f'  seam/within ratio: {ratio.mean():.2f}')

    if known is not None:
        contig = known[np.searchsorted(np.arange(n - 1)[same], idx)]
        if contig.any() and (~contig).any():
            print(f'  ratio on truly contiguous pairs: '
                  f'{ratio[contig].mean():.2f}')
            print(f'  ratio on spliced pairs:          '
                  f'{ratio[~contig].mean():.2f}')
    return ratio


print('SLEEP-EDF')
inspect('data/processed/sleep_edf_processed.h5')
same_s, contig_s = gap_test('data/processed/sleep_edf_processed.h5')
ratio_s = seam_test('data/processed/sleep_edf_processed.h5',
                    known=contig_s)

print()
print('CHB-MIT')
inspect('data/processed/chbmit_combined_seizure_detection.h5')
gap_test('data/processed/chbmit_combined_seizure_detection.h5')
ratio_c = seam_test('data/processed/chbmit_combined_seizure_detection.h5')

SLEEP-EDF
data/processed/sleep_edf_processed.h5
  keys: ['bad_epochs_mask', 'epochs', 'labels', 'subject_ids', 'times']
  attrs: {'ch_names': "['EEG Fpz-Cz', 'EEG Pz-Oz']", 'class_names': "['Wake', 'N1', 'N2', 'N3', 'REM']", 'dataset': 'sleep-edf', 'epoch_duration': 30.0, 'n_channels': 3, 'n_classes': 5, 'n_epochs': 7360, 'n_samples': 3000, 'n_subjects': 147, 'sfreq': 100}
    bad_epochs_mask: (7360,) bool
    epochs: (7360, 3, 3000) float32
    labels: (7360,) int32
    subject_ids: (7360,) |S20
    times: (7360,) float32
  within-subject adjacent pairs: 7213
  fraction exactly one epoch apart: 0.3352
  median gap: 180.0s
  pairs tested: 2000
  mean seam step: 0.8232
  mean within-epoch step: 0.2288
  seam/within ratio: 4.10
  ratio on truly contiguous pairs: 1.55
  ratio on spliced pairs:          5.42

CHB-MIT
data/processed/chbmit_combined_seizure_detection.h5
  keys: ['epochs', 'labels', 'subject_ids']
  attrs: {'dataset': 'chb-mit', 'epoch_duration': 30.0, 'mode': 'seizure_detect

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>